# 07. Демонстрация backend-слоя

Этот ноутбук показывает, как backend использует готовый production pipeline. Модель не переобучается: backend только загружает артефакт из `artifacts/movie_revenue_pipeline.joblib`, валидирует входные данные и рассчитывает бизнес-показатели.

В ноутбуке используется прямой вызов сервисного слоя, чтобы не требовать дополнительную тестовую зависимость `httpx` для FastAPI `TestClient`.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", "{:,.2f}".format)

from pydantic import ValidationError

from backend.app.schemas import MovieInput
from backend.app.services.predictor import (
    get_metrics,
    get_model_info,
    get_sample_input,
    predict_movie_success,
)

## Информация о модели и признаках

Backend отдаёт данные из `model_info.json`, `metrics.json` и `feature_schema.json`. Эти файлы фиксируют, какая модель используется и какие поля нужны для прогноза.

In [ ]:
model_info_payload = get_model_info()
metrics_payload = get_metrics()

display(pd.Series(model_info_payload["model_info"], name="model_info").to_frame())
display(pd.Series(metrics_payload, name="metrics").to_frame())

## Пример входных данных

`title` и `released` являются metadata. Они могут присутствовать в API, но не используются как признаки модели.

In [ ]:
sample_payload = get_sample_input()
sample_payload

## Прогноз через backend service

Сервис принимает `MovieInput`, формирует DataFrame для модели, вызывает `pipeline.predict()` и рассчитывает показатели коммерческого успеха.

In [ ]:
movie_input = MovieInput(**sample_payload)
prediction = predict_movie_success(movie_input)

if hasattr(prediction, "model_dump"):
    prediction_payload = prediction.model_dump()
else:
    prediction_payload = prediction.dict()

prediction_payload

## Проверка бизнес-формул

Проверим согласованность значений, которые вернул backend service.

In [ ]:
budget = sample_payload["budget"]
checks = {
    "profit_formula_ok": abs(prediction_payload["predicted_profit"] - (prediction_payload["predicted_revenue"] - budget)) < 1e-6,
    "roi_formula_ok": abs(prediction_payload["roi"] - (prediction_payload["predicted_profit"] / budget)) < 1e-6,
    "payback_formula_ok": abs(prediction_payload["payback_ratio"] - (prediction_payload["predicted_revenue"] / budget)) < 1e-6,
    "payback_equals_roi_plus_one": abs(prediction_payload["payback_ratio"] - (prediction_payload["roi"] + 1)) < 1e-6,
}
checks

## Проверка валидации бюджета

Бюджет ниже `300 000 USD` отклоняется схемой `MovieInput`, чтобы не допускать прогнозы вне устойчивой области обучающих данных.

In [ ]:
invalid_payload = dict(sample_payload)
invalid_payload["budget"] = 1

try:
    MovieInput(**invalid_payload)
except ValidationError as error:
    print("Ошибка валидации получена корректно")
    print(error)

## HTTP-запуск backend

Для ручной проверки Swagger и реального HTTP API backend запускается из корня проекта:

```bash
uvicorn backend.app.main:app --reload
```

Swagger доступен по адресу:

```text
http://127.0.0.1:8000/docs
```

## Вывод

Backend является production-обёрткой вокруг сохранённого pipeline: он не обучает модель, а выполняет загрузку, валидацию, прогноз и расчёт бизнес-показателей.